In [1]:
!pip install transformers huggingface_hub

In [2]:
!pip install ./dynamoe-0.1.0-py3-none-any.whl

Processing ./dynamoe-0.1.0-py3-none-any.whl


In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer

device_map = "auto"
model_path = "ibm-granite/granite-3.1-3b-a800m-base"
tokenizer = AutoTokenizer.from_pretrained(model_path)
# drop device_map if running on CPU
model = AutoModelForCausalLM.from_pretrained(model_path, device_map=device_map)
model.eval()
# change input text as desired
input_text = "Write 20 line poem about python language"
# tokenize the text
input_tokens = tokenizer(input_text, return_tensors="pt")

# # Send inputs to the first model parameter device for generate().
model_device = next(model.parameters()).device
input_tokens = {k: v.to(model_device) for k, v in input_tokens.items()}
# # generate output tokens
output = model.generate(**input_tokens, max_length=4000)
# # decode output tokens into text
output = tokenizer.batch_decode(output)
# # print output
print(output)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

["Write 20 line poem about python language\n\nPython, a language so divine,\nA tool for coding, a friend to many,\nIts syntax, so clean and neat,\nA language that's easy to learn,\nIts libraries, vast and wide,\nA language that's fun to code,\nA language that's loved by many,\nPython, a language so fine.<|endoftext|>"]


In [21]:
import torch, gc
from dynamoe import granite_optimizer
torch.cuda.reset_peak_memory_stats()

# print(model)
if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 2)
        curr_mem = torch.cuda.memory_allocated() / (1024 ** 2)
        print(f"🔥 Peak VRAM Allocated:   {peak_mem:.2f} MB")
        print(f"   Current VRAM Allocated: {curr_mem:.2f} MB")
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize() # Wait for all streams to finish
model = granite_optimizer(
    model,
    num_gpu_slots=3,
    eager_init=True,
    clear_cuda_cache=True,
    verbose=True,
)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
# warm-up short run so lazy buffer init happens
_ = model.generate(**input_tokens, max_new_tokens=8)
# print(model)
# gc.collect()
# torch.cuda.empty_cache()
print("after warmup alloc/res:", torch.cuda.memory_allocated()/1e9, torch.cuda.memory_reserved()/1e9)
print("peak alloc:", torch.cuda.max_memory_allocated()/1e9)
if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 2)
        curr_mem = torch.cuda.memory_allocated() / (1024 ** 2)
        print(f"🔥 Peak VRAM Allocated:   {peak_mem:.2f} MB")
        print(f"   Current VRAM Allocated: {curr_mem:.2f} MB")

🔥 Peak VRAM Allocated:   6300.07 MB
   Current VRAM Allocated: 6300.07 MB
[dynamoe] Replaced 64 GraniteMoeParallelExperts module(s) with optimized version
[dynamoe] Eager-initialized 64 optimized expert module(s)
[dynamoe] Note: monitor torch.cuda.memory_allocated() for true live usage; memory_reserved() includes allocator cache.
after warmup alloc/res: 1.01928704 7.558135808
peak alloc: 1.02105856
🔥 Peak VRAM Allocated:   973.76 MB
   Current VRAM Allocated: 972.07 MB


In [22]:
_ = model.generate(**input_tokens, max_new_tokens=8)
# print(model)
# gc.collect()
# torch.cuda.empty_cache()
print("after warmup alloc/res:", torch.cuda.memory_allocated()/1e9, torch.cuda.memory_reserved()/1e9)
print("peak alloc:", torch.cuda.max_memory_allocated()/1e9)
if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 2)
        curr_mem = torch.cuda.memory_allocated() / (1024 ** 2)
        print(f"🔥 Peak VRAM Allocated:   {peak_mem:.2f} MB")
        print(f"   Current VRAM Allocated: {curr_mem:.2f} MB")

after warmup alloc/res: 1.01928704 7.558135808
peak alloc: 1.02105856
🔥 Peak VRAM Allocated:   973.76 MB
   Current VRAM Allocated: 972.07 MB
